# Download Player Status Data

Download MLB StatsAPI transactions month-by-month (starting 2022-01-01) and cache each month as a parquet file inside parquet directories.


In [ ]:
from __future__ import annotations

from datetime import date
from pathlib import Path
import json
import urllib.error
import urllib.parse
import urllib.request

import pandas as pd


## Configuration


In [ ]:
SPORT_ID = 1
BASE_URL = "https://statsapi.mlb.com/api/v1"

# Pull a rolling 3-year window, but never before January 2022.
TODAY = date.today()
LOOKBACK_YEARS = 3
HARD_MIN_START = date(2022, 1, 1)
WINDOW_START = max(HARD_MIN_START, date(TODAY.year - LOOKBACK_YEARS + 1, 1, 1))
WINDOW_END = TODAY

TRANSACTION_TYPES = [
    "D60",   # 60-day IL
    "D10",   # 10-day IL
    "D15",   # 15-day IL variants
    "DTD",   # day-to-day
    "ACT",   # activated
    "REHAB", # rehab assignment
    "OPT",   # optioned
    "REC",   # recalled
    "DES",   # designated for assignment
    "TR",    # traded
    "SUS",   # suspended
    "RL",    # released
]

DATA_DIR = Path("data")
CACHE_ROOT = DATA_DIR / "player_status_cache"
TRANSACTIONS_DIR = CACHE_ROOT / "player_transactions"
CALL_LOG_DIR = CACHE_ROOT / "player_status_call_log"

for folder in [DATA_DIR, CACHE_ROOT, TRANSACTIONS_DIR, CALL_LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Window start: {WINDOW_START}")
print(f"Window end:   {WINDOW_END}")


## Helpers


In [ ]:
def month_windows(start_date: date, end_date: date) -> list[tuple[date, date]]:
    windows: list[tuple[date, date]] = []
    current = date(start_date.year, start_date.month, 1)

    while current <= end_date:
        if current.month == 12:
            next_month = date(current.year + 1, 1, 1)
        else:
            next_month = date(current.year, current.month + 1, 1)

        month_start = max(current, start_date)
        month_end = min(next_month.fromordinal(next_month.toordinal() - 1), end_date)
        windows.append((month_start, month_end))
        current = next_month

    return windows


def month_key(month_start: date) -> str:
    return f"{month_start.year}-{month_start.month:02d}"


def build_url(path: str, params: dict | None = None) -> str:
    params = params or {}
    query = urllib.parse.urlencode(params)
    url = f"{BASE_URL}{path}"
    return f"{url}?{query}" if query else url


def fetch_json_debug(path: str, params: dict | None = None, timeout: int = 60) -> dict:
    url = build_url(path, params)
    result = {
        "ok": False,
        "url": url,
        "status": None,
        "error": None,
        "payload": None,
        "response_text_preview": None,
    }

    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            result["status"] = response.status
            raw_text = response.read().decode("utf-8")
            result["response_text_preview"] = raw_text[:500]
            result["payload"] = json.loads(raw_text)
            result["ok"] = True
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace") if exc.fp else ""
        result["status"] = exc.code
        result["error"] = f"HTTPError: {exc}"
        result["response_text_preview"] = body[:500]
    except urllib.error.URLError as exc:
        result["error"] = f"URLError: {exc}"
    except Exception as exc:
        result["error"] = f"UnexpectedError: {exc}"

    return result


## Download + cache monthly parquet files

Each month is downloaded independently and written as a parquet file under each cache directory.


In [ ]:
windows = month_windows(WINDOW_START, WINDOW_END)

all_monthly_logs: list[pd.DataFrame] = []
written_transaction_files: list[Path] = []
written_log_files: list[Path] = []

for month_start, month_end in windows:
    monthly_frames: list[pd.DataFrame] = []
    monthly_call_results: list[dict] = []
    key = month_key(month_start)

    for type_code in TRANSACTION_TYPES:
        params = {
            "sportId": SPORT_ID,
            "startDate": month_start.isoformat(),
            "endDate": month_end.isoformat(),
            "transactionTypes": type_code,
        }
        result = fetch_json_debug("/transactions", params)
        result["type_code"] = type_code
        result["month_key"] = key
        monthly_call_results.append(result)

        if result["ok"] and result["payload"] is not None:
            records = result["payload"].get("transactions", [])
            if records:
                frame = pd.json_normalize(records)
                frame["requested_type_code"] = type_code
                frame["month_key"] = key
                monthly_frames.append(frame)

    monthly_transactions = pd.concat(monthly_frames, ignore_index=True) if monthly_frames else pd.DataFrame()

    if not monthly_transactions.empty:
        rename_map = {
            "person.id": "player_id",
            "person.fullName": "player_name",
            "toTeam.id": "to_team_id",
            "toTeam.name": "to_team_name",
            "fromTeam.id": "from_team_id",
            "fromTeam.name": "from_team_name",
            "typeCode": "type_code",
            "typeDesc": "type_desc",
            "description": "description",
            "date": "transaction_date",
            "effectiveDate": "effective_date",
        }
        monthly_transactions = monthly_transactions.rename(columns={k: v for k, v in rename_map.items() if k in monthly_transactions.columns})

        for dt_col in ["transaction_date", "effective_date"]:
            if dt_col in monthly_transactions.columns:
                monthly_transactions[dt_col] = pd.to_datetime(monthly_transactions[dt_col], errors="coerce")

        monthly_transactions = monthly_transactions.drop_duplicates(
            subset=[c for c in ["player_id", "transaction_date", "type_code", "description"] if c in monthly_transactions.columns]
        ).reset_index(drop=True)

        transaction_file = TRANSACTIONS_DIR / f"{key}.parquet"
        monthly_transactions.to_parquet(transaction_file, index=False)
        written_transaction_files.append(transaction_file)

    month_log = pd.DataFrame(
        {
            "month_key": [key for _ in monthly_call_results],
            "month_start": [month_start for _ in monthly_call_results],
            "month_end": [month_end for _ in monthly_call_results],
            "type_code": [r["type_code"] for r in monthly_call_results],
            "ok": [r["ok"] for r in monthly_call_results],
            "status": [r["status"] for r in monthly_call_results],
            "error": [r["error"] for r in monthly_call_results],
            "url": [r["url"] for r in monthly_call_results],
        }
    )

    log_file = CALL_LOG_DIR / f"{key}.parquet"
    month_log.to_parquet(log_file, index=False)
    written_log_files.append(log_file)
    all_monthly_logs.append(month_log)

print(f"Months processed: {len(windows)}")
print(f"Monthly transaction parquet files written: {len(written_transaction_files)}")
print(f"Monthly call-log parquet files written: {len(written_log_files)}")


## Load cached parquet directories

Parquet readers can load these cache folders directly as datasets.


In [ ]:
transactions = pd.read_parquet(TRANSACTIONS_DIR)
call_log = pd.read_parquet(CALL_LOG_DIR)

print(f"Cached transactions rows: {len(transactions):,}")
print(f"Cached call-log rows: {len(call_log):,}")

transactions.head(10)


In [ ]:
failed_calls = call_log.loc[~call_log["ok"]].copy()
failed_calls.head(20)


## Cache paths


In [ ]:
print("Transaction parquet directory:", TRANSACTIONS_DIR)
print("Call-log parquet directory:", CALL_LOG_DIR)
